# Companion notebook: "Cryptography Decoded": Algebra 2: Polynomial Rings and Finite Fields

© 2001–2026 Gerald Teschl (http://www.mat.univie.ac.at/~gerald/) and Susanne Teschl (http://staff.technikum-wien.at/~teschl/)

This [Python](https://python.org) notebook accompanies our textbook "[Cryptography Decoded](https://www.mat.univie.ac.at/~gerald/ftp/book-cry/)". You'll find more explanations in the book.

You can easily install the code from [PyPI](https://pypi.org/project/kryptools/).

## Polynomial rings

We'll start with a class for polynomials. Each polynomial is stored as a list of its coefficients.

The class defines addition, multiplication, and division with remainder. It also supports coefficients that are elements of a ring.

In [1]:
from kryptools import Poly

We create polynomials from a list of coefficients:

In [2]:
p = Poly([0, 1, 0, 1])
q = Poly([1, 1])
p, q

(x^3 + x, x + 1)

The following trick lets us enter polynomials in the usual form:

In [13]:
x = Poly([0,1])

x**2 + x + 1

x^2 + x + 1

Polynomial addition and multiplication:

In [3]:
p + q, p * q

(x^3 + 2 x + 1, x^4 + x^3 + x^2 + x)

Polynomial division with remainder:

In [4]:
p = Poly([0, -2, 0, 1, 3])
q = Poly([1, 0, 1])

p.divmod(q)

(3 x^2 + x - 3, -3 x + 3)

Let's check:

In [5]:
p == _[0] * q + _[1]

True

This also works:

In [6]:
p // q, p % q

(3 x^2 + x - 3, -3 x + 3)

The coefficients can come from any field (or ring). For polynomials with rational coefficients, we can use Python's `Fraction` class.

In [8]:
from fractions import Fraction

p = Poly([0, -2, 0, 1, 3], ring=Fraction)
Fraction(1 / 2) * p

3/2 x^4 + 1/2 x^3 - x

We can also change the coefficient ring later by using `map` to apply a function, such as a class constructor, to every coefficient:

In [9]:
q.map(Fraction)
q / 4

1/4 x^2 + 1/4

The coefficients can also come from $\mathbb{Z}_m$. We've already met the corresponding class, `Zmod`:

In [2]:
from kryptools import Zmod

Z_2 = Zmod(2)
p = Poly([0, -2, 0, 1, 3], ring=Z_2)
3 * p

x^4 + x^3

To find all the divisors, we need to factor the polynomial. The package can only factor polynomials over finite fields:

In [3]:
p.factor()

{x + 1: 1, x: 3}

The extended Euclidean algorithm

In [11]:
def egcd(a: Poly, b: Poly) -> (Poly, Poly, Poly):
    """Perform the extended Euclidean agorithm for polynomials. Returns gcd, x, y such that a x + b y = gcd."""
    r0, r1 = a, b
    x0, x1, y0, y1 = Poly([1]), Poly([0]), Poly([0]), Poly([1])
    while r1:
        q, r = r0.divmod(r1)
        r0, r1 = r1, r
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return r0, x0, y0


a = Poly([1, -2, 0, 1])
b = Poly([-1, 0, 1])
egcd(a, b)

(- x + 1, 1, - x)

and how we use it to calculate the multiplicative inverse:

In [12]:
def inverse(a: Poly, m: Poly) -> Poly:
    """Compute the multiplicative inverse in Z[x]_m. Returns b(x) such that a(x) * b(x) % m(x) = 1."""
    gcd, x, y = egcd(m, a)
    if gcd.degree() != 0:
        raise ValueError(f"{a} is not invertible mod {m}.")
    return Poly([1 / gcd.coeff[0]]) * y


a = Poly([0, -2, 0, 1, 3], ring=Fraction)
m = Poly([-1, 0, 1], ring=Fraction)
inverse(a, m)

1/8 x + 3/8

## Quotient rings

To reduce a polynomial $p(x)$ modulo $m(x)$, we can either calculate the remainder using `%` or use `.mod(m)`:

In [4]:
p = Poly([1, 0, 0, 1]) # x^3 + 1
m = Poly([-1, 0, 1]) # x^2 - 1
p.mod(m)
p

x + 1

Here's how to make sure all calculations involving a polynomial are carried out modulo $m(x)$:

In [14]:
Poly([1, 0, 0, 1], modulus=[-1, 0, 1])

x + 1

For example, we can create a multiplication table for the quotient ring $\mathbb{Z}_2[x]_{x^3+x+1}$:

In [8]:
from kryptools import Poly, Zmod

Z_2 = Zmod(2)
m = Poly([1,1,1], ring = Z_2)
poly_ring = [Poly([a, b], modulus = m) for b in Z_2 for a in Z_2]


[[p * q for p in poly_ring] for q in poly_ring]

[[0, 0, 0, 0], [0, 1, x, x + 1], [0, x, x + 1, 1], [0, x + 1, 1, x]]

We can calculate the multiplicative inverse as follows:

In [9]:
e = Poly([1, 1], ring = Z_2, modulus = m)
e**-1

x

## Finite fields

We can check whether a polynomial over a finite field is irreducible as follows:

In [10]:
from kryptools import Poly, Zmod

Z_2 = Zmod(2)

p = Poly([1,0,1,0], ring = Z_2)
p.is_irreducible()

False

We can also factor it:

In [11]:
p.factor()

{x + 1: 2}

For example, we can implement the quotient ring $\mathbb{Z}_2[x]_{x^8+x^4+x^3+x+1}$ used for calculations in AES:

In [18]:
from kryptools import Poly, Zmod

Z_2 = Zmod(2)
aes = [1, 1, 0, 1, 1, 0, 0, 0, 1]  # x^8 + x^4 + x^3 + x + 1


def PolyAES(c: list) -> "Poly":
    return Poly(c, ring = Z_2, modulus = aes)


p = PolyAES([1, 0, 1])  # x^2 + 1
q = PolyAES([0, 1, 0, 0, 1, 1, 1, 1])  # x^7 + x^6 + x^5 + x^4 + x

p * q

x^4 + x^2 + x + 1

Representing bytes as polynomials, with the bits as coefficients:

In [19]:
def byte2poly(x: int, n: int = 8) -> "Poly":
    "Convert an integer to a Polynomial."
    return PolyAES(list(reversed([int(d) for d in str(format(x, "0" + str(n) + "b"))])))


def poly2byte(p: "Poly") -> int:
    "Convert a polynomial to the corresponding integer."
    s = 0
    for i in reversed(p.coeff):
        s = s * 2 + int(i)
    return s


poly2byte(byte2poly(2).inv())

141

In [28]:
byte2poly(0xa3) * byte2poly(0x02)

x^6 + x^4 + x^3 + x^2 + 1

In [29]:
hex(poly2byte(_))

'0x5d'

Using the greatest common divisor of polynomials,

In [17]:
def poly_gcd(a: Poly, b: Poly) -> Poly:
    """Perform the Euclidean agorithm for polynomials."""
    r0, r1 = a, b
    while r1:
        q, r = r0.divmod(r1)
        r0, r1 = r1, r
    return r0


a = Poly([1, -2, 0, 1])  # x^3 -2 x + 1
b = Poly([-1, 0, 1])  # x^2 - 1
poly_gcd(a, b)

- x + 1

we can implement Rabin's irreducibility test:

In [18]:
def rabin_test(a: Poly, verbose: bool = False) -> bool:
    """Determine if a polynomial is irreducible over Z_p using a Rabin test."""
    gf = a[0].ring
    p = gf.n
    x = Poly([0, 1], ring=gf)  # x
    xp = Poly(p * [0] + [1], ring=gf, modulus = a.coeff)  # x^p modulo a to keep the polynomials small
    for k in range(1, a.degree() // 2 + 1):
        if k == 1:
            b = xp
        else:
            b **= p  # we compute x^{p^k} mod a recursively
        g = a.gcd(b - x)  # gcd with x^{p^k} - x
        if g.degree():
            if verbose:
                print(f"Factor found with k={k}: {g}")
            return False
    return True


a = Poly([1, 0, 1], ring=Z_2)

rabin_test(a, verbose=True)

Factor found with k=1: x + 1


False

We can use this to show that the AES polynomial is irreducible:

In [19]:
a = Poly([1, 1, 0, 1, 1, 0, 0, 0, 1], ring=Z_2)
print(a)

rabin_test(a)

x^8 + x^4 + x^3 + x + 1


True

We can express the number of irreducible polynomials of degree $k$ over the field $\mathbb{Z}_p$ using the Möbius function $\mu(n)$. For the number of irreducible polynomials of degree $8$ over the field $\mathbb{Z}_2$, we get:

In [20]:
from kryptools import divisors, moebius_mu


def M(p: int, k: int) -> int:
    """Moreau's necklace-counting function."""
    return sum([moebius_mu(d) * p ** (k // d) for d in divisors(k)]) // k


M(2, 8)

30

We can get a list of these polynomials as follows:

In [21]:
j = 0
for i in range(256):
    a = Poly(
        list(reversed([int(d) for d in str(format(256 + i, "0" + str(9) + "b"))])),
        ring=Z_2,
    )
    if rabin_test(a):
        j += 1
        print(j, ": ", a)

1 :  x^8 + x^4 + x^3 + x + 1
2 :  x^8 + x^4 + x^3 + x^2 + 1
3 :  x^8 + x^5 + x^3 + x + 1
4 :  x^8 + x^5 + x^3 + x^2 + 1
5 :  x^8 + x^5 + x^4 + x^3 + 1
6 :  x^8 + x^5 + x^4 + x^3 + x^2 + x + 1
7 :  x^8 + x^6 + x^3 + x^2 + 1
8 :  x^8 + x^6 + x^4 + x^3 + x^2 + x + 1
9 :  x^8 + x^6 + x^5 + x + 1
10 :  x^8 + x^6 + x^5 + x^2 + 1
11 :  x^8 + x^6 + x^5 + x^3 + 1
12 :  x^8 + x^6 + x^5 + x^4 + 1
13 :  x^8 + x^6 + x^5 + x^4 + x^2 + x + 1
14 :  x^8 + x^6 + x^5 + x^4 + x^3 + x + 1
15 :  x^8 + x^7 + x^2 + x + 1
16 :  x^8 + x^7 + x^3 + x + 1
17 :  x^8 + x^7 + x^3 + x^2 + 1
18 :  x^8 + x^7 + x^4 + x^3 + x^2 + x + 1
19 :  x^8 + x^7 + x^5 + x + 1
20 :  x^8 + x^7 + x^5 + x^3 + 1
21 :  x^8 + x^7 + x^5 + x^4 + 1
22 :  x^8 + x^7 + x^5 + x^4 + x^3 + x^2 + 1
23 :  x^8 + x^7 + x^6 + x + 1
24 :  x^8 + x^7 + x^6 + x^3 + x^2 + x + 1
25 :  x^8 + x^7 + x^6 + x^4 + x^2 + x + 1
26 :  x^8 + x^7 + x^6 + x^4 + x^3 + x^2 + 1
27 :  x^8 + x^7 + x^6 + x^5 + x^2 + x + 1
28 :  x^8 + x^7 + x^6 + x^5 + x^4 + x + 1
29 :  x^8 + x

### $GF(2^n)$

Since $GF(2^n)$ is particularly important in cryptography, we provide the `GF2` class for this case. Elements are represented as numbers, and their binary representations determine the corresponding polynomials.

In [12]:
from kryptools import GF2

gf = GF2(3)
[a.poly() for a in gf]

[0, 1, x, x + 1, x^2, x^2 + 1, x^2 + x, x^2 + x + 1]

An irreducible polynomial is automatically chosen from a list. Alternatively, we can specify one using the `modulus` argument when creating the field:

In [15]:
gf.poly()

x^3 + x + 1

We can use this to create, for example, an addition table, a multiplication table, and a table of multiplicative inverses:

In [24]:
[[a + b for a in gf] for b in gf]

[[0, 1, 2, 3, 4, 5, 6, 7],
 [1, 0, 3, 2, 5, 4, 7, 6],
 [2, 3, 0, 1, 6, 7, 4, 5],
 [3, 2, 1, 0, 7, 6, 5, 4],
 [4, 5, 6, 7, 0, 1, 2, 3],
 [5, 4, 7, 6, 1, 0, 3, 2],
 [6, 7, 4, 5, 2, 3, 0, 1],
 [7, 6, 5, 4, 3, 2, 1, 0]]

In [25]:
[[a * b for a in gf] for b in gf]

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 2, 3, 4, 5, 6, 7],
 [0, 2, 4, 6, 3, 1, 7, 5],
 [0, 3, 6, 5, 7, 4, 1, 2],
 [0, 4, 3, 7, 6, 2, 5, 1],
 [0, 5, 1, 4, 2, 7, 3, 6],
 [0, 6, 7, 1, 5, 3, 2, 4],
 [0, 7, 5, 2, 1, 6, 4, 3]]

In [26]:
[a ** -1 for a in gf.star()]

[1, 5, 6, 7, 2, 3, 4]